# LeetCode 581 — Shortest Unsorted Continuous Subarray bằng Python

Link đề: https://leetcode.com/problems/shortest-unsorted-continuous-subarray/ · Độ khó: **Medium** · Chủ đề: Array, Two Pointers, Stack, Greedy, Sorting, Monotonic Stack

Skeleton LeetCode:

```python
class Solution:
    def findUnsortedSubarray(self, nums: List[int]) -> int:
        ...
```

Ghi chú: bài này không cần cấu trúc tự dựng nào (chỉ có mảng số nguyên). Câu hỏi nâng cao (follow up) của đề: giải trong **O(n)**. Notebook này đi qua 5 cách: vét cạn O(n²), sắp xếp rồi so sánh, ngăn xếp đơn điệu, "max bên trái / min bên phải" (hai lượt quét), và "tìm đoạn lõm rồi nở ra".

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tìm **đoạn liên tiếp ngắn nhất** mà nếu **chỉ sắp xếp đoạn đó** thì **cả mảng** trở thành tăng dần (không giảm), rồi trả về **độ dài** của đoạn.
2. Nắm các công cụ Python cần dùng: `sorted()` khác `.sort()` thế nào, cắt lát (slice) `nums[i:j+1]`, `zip`/`enumerate` để so sánh hai list, `float('inf')` làm giá trị canh, `list` làm **ngăn xếp (stack)**, và quét **từ phải sang trái** bằng `range(n - 1, -1, -1)`.
3. Đi từ **cặp nghịch thế O(n²)** → **sắp xếp + so sánh O(n log n)** → **ngăn xếp đơn điệu O(n)** → **hai lượt quét max/min O(n), O(1) bộ nhớ** → **tìm đoạn lõm rồi nở ra**.
4. Hiểu mấu chốt: một phần tử nằm **đúng chỗ** khi và chỉ khi nó **không nhỏ hơn max bên trái** và **không lớn hơn min bên phải**; đoạn cần sắp xếp chạy từ phần tử "sai chỗ" **đầu tiên** đến phần tử "sai chỗ" **cuối cùng**.
5. Tránh các lỗi kinh điển: chỉ nhìn các cặp **kề nhau**, khởi tạo `l = r = 0` làm mảng đã sắp xếp trả về 1, dùng `<=` thay vì `<` khi có **số trùng**, sắp xếp **tại chỗ** rồi so sánh với chính nó, và quên **nở** đoạn lõm ra hai phía.

## 2. Đề bài & ví dụ

Cho một mảng số nguyên `nums`. Bạn cần tìm **một mảng con liên tiếp** sao cho **nếu chỉ sắp xếp mảng con đó theo thứ tự không giảm** thì **toàn bộ mảng** cũng được sắp xếp theo thứ tự không giảm.

Hãy trả về **độ dài của mảng con ngắn nhất** như vậy.

**Ví dụ 1**

- Input: `nums = [2,6,4,8,10,9,15]`
- Output: `5`
- Giải thích: cần sắp xếp `[6, 4, 8, 10, 9]` (từ chỉ số 1 đến 5) thì cả mảng thành `[2,4,6,8,9,10,15]`.

**Ví dụ 2**

- Input: `nums = [1,2,3,4]`
- Output: `0`
- Giải thích: mảng đã sắp xếp rồi, không cần sắp xếp đoạn nào nên độ dài là `0`.

**Ví dụ 3**

- Input: `nums = [1]`
- Output: `0`

**Ràng buộc:**

- `1 <= nums.length <= 10^4`
- `-10^5 <= nums[i] <= 10^5`

**Follow up:** bạn có thể giải trong thời gian `O(n)` không?

Chú ý: "sắp xếp **không giảm**" nghĩa là cho phép các số **bằng nhau** đứng cạnh nhau (`[1,2,2,3]` là đã sắp xếp). Điều này rất quan trọng khi mảng có số trùng.

In [1]:
from typing import List
import random
import time
from itertools import accumulate

examples = [
    ([2, 6, 4, 8, 10, 9, 15], 5),
    ([1, 2, 3, 4], 0),
    ([1], 0),
]
for nums, expected in examples:
    print(nums, "-> kỳ vọng", expected)

[2, 6, 4, 8, 10, 9, 15] -> kỳ vọng 5
[1, 2, 3, 4] -> kỳ vọng 0
[1] -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1. `sorted()` khác `.sort()`

- `sorted(a)` **trả về một list mới** đã sắp xếp, `a` giữ nguyên.
- `a.sort()` sắp xếp **ngay trong `a`** và **trả về `None`** (không phải list!).

Trong bài này ta thường cần **giữ nguyên** `nums` để so sánh với bản đã sắp xếp, nên dùng `sorted()`.

In [2]:
a = [3, 1, 2]
b = sorted(a)
print("a =", a, "| b = sorted(a) =", b)

c = a.sort()
print("sau a.sort(): a =", a, "| giá trị trả về =", c)

a = [3, 1, 2] | b = sorted(a) = [1, 2, 3]
sau a.sort(): a = [1, 2, 3] | giá trị trả về = None


### 3.2. Cắt lát (slice) `nums[i:j+1]`

`nums[i:j+1]` lấy các phần tử có chỉ số `i, i+1, ..., j` (chỉ số cuối `j+1` **không** được lấy). Ta dùng nó để lấy một đoạn liên tiếp rồi gọi `min(...)`, `max(...)`, `sorted(...)` trên đoạn đó.

In [3]:
nums = [2, 6, 4, 8, 10, 9, 15]
seg = nums[1:6]          # chỉ số 1..5
print("đoạn 1..5 =", seg)
print("min =", min(seg), "| max =", max(seg), "| sorted =", sorted(seg))

# ghép lại mảng sau khi chỉ sắp xếp đoạn 1..5:
after = nums[:1] + sorted(nums[1:6]) + nums[6:]
print("sau khi sắp xếp đoạn:", after)

đoạn 1..5 = [6, 4, 8, 10, 9]
min = 4 | max = 10 | sorted = [4, 6, 8, 9, 10]
sau khi sắp xếp đoạn: [2, 4, 6, 8, 9, 10, 15]


### 3.3. `zip` và `enumerate` để so sánh hai list theo từng vị trí

`zip(x, y)` ghép phần tử cùng vị trí của hai list thành từng cặp. `enumerate(x)` thêm chỉ số. Hai list bằng nhau khi so bằng toán tử `==` (so từng phần tử theo thứ tự).

In [4]:
x = [2, 6, 4, 8]
y = sorted(x)
print("y =", y)
for i, (p, q) in enumerate(zip(x, y)):
    print(f"vị trí {i}: x={p}, y={q}, {'giống' if p == q else 'KHÁC'}")
print("x == y ?", x == y, "| y == sorted(y) ?", y == sorted(y))

y = [2, 4, 6, 8]
vị trí 0: x=2, y=2, giống
vị trí 1: x=6, y=4, KHÁC
vị trí 2: x=4, y=6, KHÁC
vị trí 3: x=8, y=8, giống
x == y ? False | y == sorted(y) ? True


### 3.4. Giá trị canh `float('inf')`

`float('inf')` là "vô cực dương" — lớn hơn mọi số nguyên; `-float('inf')` nhỏ hơn mọi số nguyên. Ta dùng chúng để khởi tạo "max đến giờ" (bắt đầu là `-inf`) và "min đến giờ" (bắt đầu là `+inf`) mà không cần xử lý riêng phần tử đầu tiên.

In [5]:
inf = float('inf')
print(5 < inf, -10**9 > -inf)

mx = -inf
for v in [3, 7, 2]:
    mx = max(mx, v)
    print("sau khi thấy", v, "-> max đến giờ =", mx)

True True
sau khi thấy 3 -> max đến giờ = 3
sau khi thấy 7 -> max đến giờ = 7
sau khi thấy 2 -> max đến giờ = 7


### 3.5. `list` làm ngăn xếp (stack) và vòng `while stk and ...`

Ngăn xếp: vào sau – ra trước (LIFO). Với `list`: `stk.append(x)` đẩy vào đỉnh, `stk.pop()` lấy đỉnh ra, `stk[-1]` xem đỉnh. Luôn kiểm tra `stk` **không rỗng** trước khi đọc `stk[-1]`, nên viết `while stk and điều_kiện:` (Python dừng ngay nếu `stk` rỗng nhờ cơ chế *short-circuit*).

In [6]:
stk = []
stk.append(5); stk.append(8); stk.append(3)
print("stack:", stk, "| đỉnh:", stk[-1])
print("pop ->", stk.pop(), "| còn:", stk)

# bỏ các phần tử lớn hơn 4 ở đỉnh:
stk = [1, 3, 6, 9]
while stk and stk[-1] > 4:
    print("bỏ", stk.pop())
print("còn lại:", stk)

stack: [5, 8, 3] | đỉnh: 3
pop -> 3 | còn: [5, 8]
bỏ 9
bỏ 6
còn lại: [1, 3]


### 3.6. Quét ngược: `range(n - 1, -1, -1)`

`range(start, stop, step)` với `step = -1` đếm lùi. `range(n - 1, -1, -1)` cho `n-1, n-2, ..., 0` (dừng **trước** `-1`). Đây là cách duyệt từ phải sang trái mà vẫn có chỉ số `i` để gán `l = i`.

In [7]:
n = 5
print(list(range(n - 1, -1, -1)))
nums = [10, 20, 30, 40, 50]
for i in range(n - 1, -1, -1):
    print("i =", i, "nums[i] =", nums[i])

[4, 3, 2, 1, 0]
i = 4 nums[i] = 50
i = 3 nums[i] = 40
i = 2 nums[i] = 30
i = 1 nums[i] = 20
i = 0 nums[i] = 10


### 3.7. Prefix max (max chạy từ trái) bằng `itertools.accumulate`

"Prefix max" tại `i` là `max(nums[0..i])`. Có thể tính bằng vòng lặp hoặc `accumulate(nums, max)`. Tương tự **suffix min** tại `i` là `min(nums[i..n-1])` — tính từ phải sang trái. Hai mảng này sẽ là "linh hồn" của lời giải O(n).

In [8]:
nums = [2, 6, 4, 8, 10, 9, 15]
prefix_max = list(accumulate(nums, max))
print("nums       =", nums)
print("prefix_max =", prefix_max)

suffix_min = list(accumulate(reversed(nums), min))[::-1]
print("suffix_min =", suffix_min)

nums       = [2, 6, 4, 8, 10, 9, 15]
prefix_max = [2, 6, 6, 8, 10, 10, 15]
suffix_min = [2, 4, 4, 8, 9, 9, 15]


## 4. Ý tưởng nền tảng / trực giác

Nếu mảng đã sắp xếp, mỗi phần tử đã **đúng chỗ**. Khi mảng chưa sắp xếp, ta hình dung mảng đã sắp xếp (gọi là `sorted_nums`) và đặt **cạnh** mảng gốc. Phần tử nào **khác** vị trí tương ứng là phần tử **sai chỗ**. Đoạn cần sắp xếp chắc chắn phải phủ **mọi** vị trí sai chỗ.

Tính tay với `nums = [2,6,4,8,10,9,15]`:

| chỉ số | 0 | 1 | 2 | 3 | 4 | 5 | 6 |
|---|---|---|---|---|---|---|---|
| `nums` | 2 | 6 | 4 | 8 | 10 | 9 | 15 |
| `sorted` | 2 | 4 | 6 | 8 | 9 | 10 | 15 |
| khác? | | khác | khác | | khác | khác | |

Vị trí khác đầu tiên là `1`, cuối cùng là `5`, nên đoạn là `[1..5]`, độ dài `5 - 1 + 1 = 5`. Ta thấy cả vị trí `3` (giá trị `8`, đúng chỗ) **nằm giữa** đoạn nhưng vẫn phải sắp xếp cùng — vì đoạn phải **liên tiếp**.

Hãy để máy làm đúng phép so sánh này.

In [9]:
nums = [2, 6, 4, 8, 10, 9, 15]
srt = sorted(nums)
diff = [i for i in range(len(nums)) if nums[i] != srt[i]]
print("nums  =", nums)
print("sorted=", srt)
print("các vị trí sai chỗ:", diff)
print("đoạn = [", diff[0], "..", diff[-1], "] -> độ dài", diff[-1] - diff[0] + 1)

nums  = [2, 6, 4, 8, 10, 9, 15]
sorted= [2, 4, 6, 8, 9, 10, 15]
các vị trí sai chỗ: [1, 2, 4, 5]
đoạn = [ 1 .. 5 ] -> độ dài 5


### Một "trọng tài" vét cạn cực chậm nhưng chắc chắn đúng

Để sau này **kiểm chứng** các lời giải thông minh, ta viết một hàm `oracle` làm đúng theo đề, không suy luận gì: thử **mọi đoạn** `[i..j]`, sắp xếp đoạn đó, ghép lại, xem cả mảng có tăng dần không; lấy độ dài nhỏ nhất. Độ phức tạp khoảng O(n³ log n) — chỉ dùng cho mảng nhỏ.

In [10]:
def oracle(nums):
    """Trọng tài: thử mọi đoạn [i..j], sắp xếp đoạn, kiểm tra cả mảng."""
    n = len(nums)
    target = sorted(nums)
    if nums == target:
        return 0
    best = n                      # sắp xếp cả mảng chắc chắn đúng
    for i in range(n):
        for j in range(i, n):
            cand = nums[:i] + sorted(nums[i:j + 1]) + nums[j + 1:]
            if cand == target:
                best = min(best, j - i + 1)
    return best

for nums, expected in examples:
    assert oracle(nums) == expected
    print(nums, "->", oracle(nums))

[2, 6, 4, 8, 10, 9, 15] -> 5
[1, 2, 3, 4] -> 0
[1] -> 0


## 5. Cách 1 — Brute Force: tìm các cặp nghịch thế (O(n²))

**Ý tưởng.** Một cặp `(i, j)` với `i < j` mà `nums[i] > nums[j]` gọi là một **cặp nghịch thế** (inversion): số lớn đứng trước số nhỏ — sai thứ tự. Nếu cả hai cùng nằm **ngoài** đoạn sắp xếp, thứ tự sai đó vẫn còn → mảng không thể tăng dần. Vì vậy:

- Đoạn phải phủ **ít nhất một đầu** của mọi cặp nghịch thế; và (sẽ chứng minh ở mục 10) khi phủ một đầu thì phải phủ **cả hai đầu**.
- Do đó đoạn ngắn nhất bắt đầu ở `l = min(i)` và kết thúc ở `r = max(j)` trên mọi cặp nghịch thế `(i, j)`.

Code: hai vòng lặp lồng nhau tìm mọi cặp, cập nhật `left` nhỏ nhất và `right` lớn nhất.

In [11]:
def brute_force(nums):
    n = len(nums)
    left, right = n, -1          # left càng nhỏ càng tốt, right càng lớn càng tốt
    for i in range(n):
        for j in range(i + 1, n):
            if nums[i] > nums[j]:            # cặp nghịch thế
                left = min(left, i)
                right = max(right, j)
    return 0 if right == -1 else right - left + 1

for nums, expected in examples:
    assert brute_force(nums) == expected
    print(nums, "->", brute_force(nums))

[2, 6, 4, 8, 10, 9, 15] -> 5
[1, 2, 3, 4] -> 0
[1] -> 0


**Độ phức tạp.** Hai vòng lồng nhau trên `n` phần tử: **Time O(n²)**, **Space O(1)**. Với `n = 10^4` là khoảng `5 × 10^7` phép so sánh — Python thuần có thể chạy vài giây, rủi ro quá thời gian.

**Nút thắt.** Ta so sánh **mọi cặp** trong khi chỉ cần biết hai thứ: vị trí sai chỗ **đầu tiên** và **cuối cùng**. Muốn biết `nums[i]` có sai chỗ không, ta không cần xem từng phần tử bên phải — chỉ cần biết **min** của chúng; tương tự ta chỉ cần **max** bên trái. Đó là bước đột phá ở mục kế tiếp.

## 6. Ý tưởng thuật toán / bước đột phá

Hãy định nghĩa: phần tử `nums[i]` **đúng chỗ** nếu

> `max(nums[0..i-1]) <= nums[i] <= min(nums[i+1..n-1])`

(mọi thứ bên trái không lớn hơn nó, mọi thứ bên phải không nhỏ hơn nó). Nếu không đúng chỗ thì `nums[i]` phải nằm trong đoạn cần sắp xếp.

Từ đó:

- **Biên phải `r`**: vị trí **cuối cùng** `i` có `nums[i] < max(nums[0..i-1])` (có số lớn hơn nó nằm bên trái).
- **Biên trái `l`**: vị trí **đầu tiên** `i` có `nums[i] > min(nums[i+1..n-1])` (có số nhỏ hơn nó nằm bên phải).

Lượt quét trái → phải giữ `mx` = max đến giờ: nếu `nums[i] < mx` thì `i` có số lớn hơn ở bên trái → cập nhật `r = i` (lần cập nhật cuối cùng cho `r` lớn nhất). Lượt quét phải → trái giữ `mi` = min đến giờ: nếu `nums[i] > mi` thì cập nhật `l = i` (lần cuối chính là `l` nhỏ nhất).

Hãy tính các giá trị trung gian bằng mảng `prefix_max` và `suffix_min` cho ví dụ 1 trước khi tối ưu bộ nhớ.

In [12]:
nums = [2, 6, 4, 8, 10, 9, 15]
n = len(nums)
prefix_max = list(accumulate(nums, max))
suffix_min = list(accumulate(reversed(nums), min))[::-1]

print("i          :", list(range(n)))
print("nums       :", nums)
print("prefix_max :", prefix_max)
print("suffix_min :", suffix_min)

# r: vị trí cuối i mà nums[i] < max bên trái (prefix_max[i-1])
r = max((i for i in range(1, n) if nums[i] < prefix_max[i - 1]), default=-1)
# l: vị trí đầu i mà nums[i] > min bên phải (suffix_min[i+1])
l = min((i for i in range(n - 1) if nums[i] > suffix_min[i + 1]), default=-1)
print("l =", l, "| r =", r, "| độ dài =", r - l + 1)

i          : [0, 1, 2, 3, 4, 5, 6]
nums       : [2, 6, 4, 8, 10, 9, 15]
prefix_max : [2, 6, 6, 8, 10, 10, 15]
suffix_min : [2, 4, 4, 8, 9, 9, 15]
l = 1 | r = 5 | độ dài = 5


Kết quả khớp: `l = 1`, `r = 5`, độ dài `5`. Bây giờ chuyển ý tưởng này thành các cách giải gọn hơn.

## 7. Các cách tiếp theo

Ta có 5 cách tổng cộng (cách 1 đã làm ở mục 5):

| # | Cách | Time | Space |
|---|---|---|---|
| 1 | Cặp nghịch thế (brute force) | O(n²) | O(1) |
| 2 | Sắp xếp + so sánh | O(n log n) | O(n) |
| 3 | Ngăn xếp đơn điệu | O(n) | O(n) |
| 4 | Max trái / Min phải (hai lượt quét) | O(n) | O(1) |
| 5 | Tìm đoạn lõm rồi nở ra | O(n) | O(1) |

### 7.1. Cách 2 — Sắp xếp rồi so sánh

**Giải thích từng đoạn.**

1. `arr = sorted(nums)`: bản đã sắp xếp (đây là "đáp án đích" của cả mảng).
2. Hai con trỏ `l = 0`, `r = n - 1`. Dịch `l` sang phải **chừng nào** `nums[l] == arr[l]` (phần tử đầu đã đúng chỗ, không cần đụng).
3. Dịch `r` sang trái **chừng nào** `nums[r] == arr[r]`.
4. Điều kiện `l <= r` trong cả hai vòng để không vượt biên khi mảng đã sắp xếp (khi đó `l` chạy qua `r`, và `r - l + 1 = 0`).
5. Trả `r - l + 1`.

Lưu ý: cả đoạn `[l..r]` phải sắp xếp, kể cả những phần tử ở giữa đã trùng `arr`, vì đoạn phải liên tiếp.

In [13]:
def sort_compare(nums):
    arr = sorted(nums)
    l, r = 0, len(nums) - 1
    while l <= r and nums[l] == arr[l]:
        l += 1
    while l <= r and nums[r] == arr[r]:
        r -= 1
    return r - l + 1

for nums, expected in examples:
    assert sort_compare(nums) == expected
    print(nums, "->", sort_compare(nums))

[2, 6, 4, 8, 10, 9, 15] -> 5
[1, 2, 3, 4] -> 0
[1] -> 0


In [14]:
# Code nộp LeetCode — Cách 2: sắp xếp + so sánh
class Solution:
    def findUnsortedSubarray(self, nums: List[int]) -> int:
        arr = sorted(nums)
        l, r = 0, len(nums) - 1
        while l <= r and nums[l] == arr[l]:
            l += 1
        while l <= r and nums[r] == arr[r]:
            r -= 1
        return r - l + 1

for nums, expected in examples:
    assert Solution().findUnsortedSubarray(nums) == expected
print("Cách 2 qua ví dụ đề")

Cách 2 qua ví dụ đề


**Độ phức tạp.** Sắp xếp O(n log n), hai vòng quét O(n) → **Time O(n log n)**. Bản sao `arr` → **Space O(n)**. Với `n ≤ 10^4` hoàn toàn đủ nhanh, nhưng chưa đạt follow up O(n).

### 7.2. Cách 3 — Ngăn xếp đơn điệu (Monotonic Stack)

**Ý tưởng.** Ta tìm `l` = **vị trí nhỏ nhất** là đầu *trái* của một cặp nghịch thế, và `r` = **vị trí lớn nhất** là đầu *phải*.

**Lượt 1 (tìm `l`), duyệt trái → phải:** giữ một ngăn xếp các **chỉ số** sao cho giá trị tương ứng **không giảm** từ đáy lên đỉnh. Khi gặp `nums[i]` nhỏ hơn giá trị ở đỉnh, đỉnh đó (số lớn đứng trước số nhỏ) là đầu trái của một cặp nghịch thế → **pop** và cập nhật `l = min(l, chỉ_số_pop)`. Sau đó đẩy `i` vào.

**Lượt 2 (tìm `r`), duyệt phải → trái:** ngăn xếp có giá trị **không tăng** từ đáy lên đỉnh (khi nhìn theo chiều duyệt). Khi `nums[i]` **lớn hơn** giá trị ở đỉnh, đỉnh đó là đầu phải của cặp nghịch thế → pop, `r = max(r, chỉ_số_pop)`.

Dùng so sánh **nghiêm ngặt** (`>` và `<`) để các số **bằng nhau** không bị coi là sai thứ tự.

In [15]:
def stack_way(nums):
    n = len(nums)
    l, r = n, -1
    stk = []
    for i in range(n):                                  # lượt 1: tìm l
        while stk and nums[stk[-1]] > nums[i]:
            l = min(l, stk.pop())
        stk.append(i)
    stk = []
    for i in range(n - 1, -1, -1):                      # lượt 2: tìm r
        while stk and nums[stk[-1]] < nums[i]:
            r = max(r, stk.pop())
        stk.append(i)
    return 0 if r == -1 else r - l + 1

for nums, expected in examples:
    assert stack_way(nums) == expected
    print(nums, "->", stack_way(nums))

[2, 6, 4, 8, 10, 9, 15] -> 5
[1, 2, 3, 4] -> 0
[1] -> 0


In [16]:
# Code nộp LeetCode — Cách 3: ngăn xếp đơn điệu
class Solution:
    def findUnsortedSubarray(self, nums: List[int]) -> int:
        n = len(nums)
        l, r = n, -1
        stk = []
        for i in range(n):
            while stk and nums[stk[-1]] > nums[i]:
                l = min(l, stk.pop())
            stk.append(i)
        stk = []
        for i in range(n - 1, -1, -1):
            while stk and nums[stk[-1]] < nums[i]:
                r = max(r, stk.pop())
            stk.append(i)
        return 0 if r == -1 else r - l + 1

for nums, expected in examples:
    assert Solution().findUnsortedSubarray(nums) == expected
print("Cách 3 qua ví dụ đề")

Cách 3 qua ví dụ đề


**Độ phức tạp.** Mỗi chỉ số vào ngăn xếp đúng một lần và ra tối đa một lần ở mỗi lượt → **Time O(n)**. Ngăn xếp tối đa `n` phần tử → **Space O(n)**.

### 7.3. Cách 4 — Max bên trái / Min bên phải (hai lượt quét, O(1) bộ nhớ)

Đây là cách trực tiếp từ mục 6, nhưng **không** lưu mảng `prefix_max`/`suffix_min`; chỉ giữ hai biến chạy `mx` và `mi`.

**Giải thích từng đoạn.**

1. `mx = -inf`, `r = -1`. Duyệt `i` từ trái sang phải: nếu `nums[i] < mx` thì có số lớn hơn nằm bên trái → `i` sai chỗ → `r = i`; ngược lại (`nums[i] >= mx`) cập nhật `mx = nums[i]`.
2. `mi = +inf`, `l = -1`. Duyệt `i` từ phải sang trái: nếu `nums[i] > mi` thì có số nhỏ hơn bên phải → `l = i`; ngược lại `mi = nums[i]`.
3. Nếu `r == -1` (không phần tử nào sai chỗ) thì mảng đã sắp xếp → trả `0`; còn lại trả `r - l + 1`.

Chú ý **dấu**: dùng `<` / `>` nghiêm ngặt, vì phần tử **bằng** max/min vẫn đúng chỗ (không giảm cho phép bằng nhau).

In [17]:
def two_pass(nums):
    n = len(nums)
    inf = float('inf')
    mx, r = -inf, -1
    for i in range(n):                  # trái -> phải
        if nums[i] < mx:
            r = i
        else:
            mx = nums[i]
    mi, l = inf, -1
    for i in range(n - 1, -1, -1):      # phải -> trái
        if nums[i] > mi:
            l = i
        else:
            mi = nums[i]
    return 0 if r == -1 else r - l + 1

for nums, expected in examples:
    assert two_pass(nums) == expected
    print(nums, "->", two_pass(nums))

[2, 6, 4, 8, 10, 9, 15] -> 5
[1, 2, 3, 4] -> 0
[1] -> 0


In [18]:
# Code nộp LeetCode — Cách 4: max trái / min phải (hai lượt quét)
class Solution:
    def findUnsortedSubarray(self, nums: List[int]) -> int:
        n = len(nums)
        inf = float('inf')
        mx, r = -inf, -1
        for i in range(n):
            if nums[i] < mx:
                r = i
            else:
                mx = nums[i]
        mi, l = inf, -1
        for i in range(n - 1, -1, -1):
            if nums[i] > mi:
                l = i
            else:
                mi = nums[i]
        return 0 if r == -1 else r - l + 1

for nums, expected in examples:
    assert Solution().findUnsortedSubarray(nums) == expected
print("Cách 4 qua ví dụ đề")

Cách 4 qua ví dụ đề


**Độ phức tạp.** Hai lượt, mỗi lượt `n` bước → **Time O(n)**. Chỉ vài biến → **Space O(1)**. Đây là lời giải ngắn, nhanh và đạt follow up; nó cũng là cách ta chọn để nộp ở mục 9.

### 7.4. Cách 5 — Tìm đoạn lõm rồi nở ra (min/max của đoạn lõm)

**Ý tưởng khác hẳn các cách trên:** không dùng max/min chạy, mà làm theo 3 bước "người":

1. **Tìm đoạn lõm.** Từ đầu mảng, đi tới khi gặp lần đầu `nums[i] > nums[i+1]` (chỗ đầu tiên bị giảm); từ cuối mảng đi lùi tới khi gặp lần đầu `nums[j-1] > nums[j]`. Nếu không có chỗ giảm nào → mảng đã sắp xếp → `0`. Đoạn `[i..j]` là "lõm" chắc chắn phải sắp xếp.
2. **Tìm min và max** của đoạn lõm: `lo = min(nums[i..j])`, `hi = max(nums[i..j])`.
3. **Nở ra.** Phần tiền tố bên trái `i` đã tăng dần; những phần tử `nums[i-1]` mà **lớn hơn `lo`** thì lẽ ra phải đứng sau `lo` → phải kéo vào đoạn: `while i > 0 and nums[i-1] > lo: i -= 1`. Tương tự bên phải: `while j < n-1 and nums[j+1] < hi: j += 1`.

Vì sao chỉ cần `lo`, `hi` của đoạn lõm ban đầu (không cập nhật lại khi nở)? Phần tử được kéo vào bên trái đứng trong phần đã tăng dần nên nó `<= nums[i] <= hi`; và nó `> lo` nên không làm đổi `lo`. Tương tự bên phải. Ta kiểm chứng bằng test ở mục 12.

In [19]:
def dip_expand(nums):
    n = len(nums)
    i = 0
    while i + 1 < n and nums[i] <= nums[i + 1]:
        i += 1
    if i == n - 1:                       # không có chỗ giảm nào
        return 0
    j = n - 1
    while j > 0 and nums[j - 1] <= nums[j]:
        j -= 1
    seg = nums[i:j + 1]
    lo, hi = min(seg), max(seg)
    while i > 0 and nums[i - 1] > lo:
        i -= 1
    while j < n - 1 and nums[j + 1] < hi:
        j += 1
    return j - i + 1

for nums, expected in examples:
    assert dip_expand(nums) == expected
    print(nums, "->", dip_expand(nums))

[2, 6, 4, 8, 10, 9, 15] -> 5
[1, 2, 3, 4] -> 0
[1] -> 0


In [20]:
# Code nộp LeetCode — Cách 5: tìm đoạn lõm rồi nở ra
class Solution:
    def findUnsortedSubarray(self, nums: List[int]) -> int:
        n = len(nums)
        i = 0
        while i + 1 < n and nums[i] <= nums[i + 1]:
            i += 1
        if i == n - 1:
            return 0
        j = n - 1
        while j > 0 and nums[j - 1] <= nums[j]:
            j -= 1
        seg = nums[i:j + 1]
        lo, hi = min(seg), max(seg)
        while i > 0 and nums[i - 1] > lo:
            i -= 1
        while j < n - 1 and nums[j + 1] < hi:
            j += 1
        return j - i + 1

for nums, expected in examples:
    assert Solution().findUnsortedSubarray(nums) == expected
print("Cách 5 qua ví dụ đề")

Cách 5 qua ví dụ đề


**Độ phức tạp.** Mỗi con trỏ chỉ đi một chiều và `min`/`max` trên đoạn tối đa `n` phần tử → **Time O(n)**. Lát cắt `nums[i:j+1]` tạo một list con nên **Space O(n)** trong bản viết trên; có thể đổi thành vòng `for` tìm min/max để còn **O(1)** nếu cần (một bài tập ở mục 15).

## 8. TRACE từng bước

Trace **cách 4** (hai lượt quét) với `nums = [2,6,4,8,10,9,15]`, `n = 7`, kỳ vọng `5`.

**Lượt 1 — trái → phải** (giữ `mx`, cập nhật `r`):

| i | nums[i] | mx trước | so sánh | hành động | mx sau | r |
|---|---|---|---|---|---|---|
| 0 | 2 | -inf | 2 < -inf? không | mx = 2 | 2 | -1 |
| 1 | 6 | 2 | 6 < 2? không | mx = 6 | 6 | -1 |
| 2 | 4 | 6 | 4 < 6? **có** | r = 2 | 6 | 2 |
| 3 | 8 | 6 | không | mx = 8 | 8 | 2 |
| 4 | 10 | 8 | không | mx = 10 | 10 | 2 |
| 5 | 9 | 10 | 9 < 10? **có** | r = 5 | 10 | 5 |
| 6 | 15 | 10 | không | mx = 15 | 15 | 5 |

**Lượt 2 — phải → trái** (giữ `mi`, cập nhật `l`):

| i | nums[i] | mi trước | so sánh | hành động | mi sau | l |
|---|---|---|---|---|---|---|
| 6 | 15 | +inf | 15 > inf? không | mi = 15 | 15 | -1 |
| 5 | 9 | 15 | 9 > 15? không | mi = 9 | 9 | -1 |
| 4 | 10 | 9 | 10 > 9? **có** | l = 4 | 9 | 4 |
| 3 | 8 | 9 | không | mi = 8 | 8 | 4 |
| 2 | 4 | 8 | không | mi = 4 | 4 | 4 |
| 1 | 6 | 4 | 6 > 4? **có** | l = 1 | 4 | 1 |
| 0 | 2 | 4 | không | mi = 2 | 2 | 1 |

Kết quả `r - l + 1 = 5 - 1 + 1 = 5`. Hãy để hàm `trace_two_pass` in lại đúng các bước này.

In [21]:
def trace_two_pass(nums):
    n = len(nums)
    inf = float('inf')
    print("nums =", nums)
    print("--- lượt 1: trái -> phải ---")
    mx, r = -inf, -1
    for i in range(n):
        if nums[i] < mx:
            r = i
            act = f"nums[i] < mx  -> r = {i}"
        else:
            mx = nums[i]
            act = f"nums[i] >= mx -> mx = {mx}"
        print(f"i={i} nums[i]={nums[i]:>3} | {act:<24} | mx={mx} r={r}")
    print("--- lượt 2: phải -> trái ---")
    mi, l = inf, -1
    for i in range(n - 1, -1, -1):
        if nums[i] > mi:
            l = i
            act = f"nums[i] > mi  -> l = {i}"
        else:
            mi = nums[i]
            act = f"nums[i] <= mi -> mi = {mi}"
        print(f"i={i} nums[i]={nums[i]:>3} | {act:<24} | mi={mi} l={l}")
    ans = 0 if r == -1 else r - l + 1
    print(f"l = {l}, r = {r} -> độ dài {ans}")
    return ans

assert trace_two_pass([2, 6, 4, 8, 10, 9, 15]) == 5

nums = [2, 6, 4, 8, 10, 9, 15]
--- lượt 1: trái -> phải ---
i=0 nums[i]=  2 | nums[i] >= mx -> mx = 2  | mx=2 r=-1
i=1 nums[i]=  6 | nums[i] >= mx -> mx = 6  | mx=6 r=-1
i=2 nums[i]=  4 | nums[i] < mx  -> r = 2   | mx=6 r=2
i=3 nums[i]=  8 | nums[i] >= mx -> mx = 8  | mx=8 r=2
i=4 nums[i]= 10 | nums[i] >= mx -> mx = 10 | mx=10 r=2
i=5 nums[i]=  9 | nums[i] < mx  -> r = 5   | mx=10 r=5
i=6 nums[i]= 15 | nums[i] >= mx -> mx = 15 | mx=15 r=5
--- lượt 2: phải -> trái ---
i=6 nums[i]= 15 | nums[i] <= mi -> mi = 15 | mi=15 l=-1
i=5 nums[i]=  9 | nums[i] <= mi -> mi = 9  | mi=9 l=-1
i=4 nums[i]= 10 | nums[i] > mi  -> l = 4   | mi=9 l=4
i=3 nums[i]=  8 | nums[i] <= mi -> mi = 8  | mi=8 l=4
i=2 nums[i]=  4 | nums[i] <= mi -> mi = 4  | mi=4 l=4
i=1 nums[i]=  6 | nums[i] > mi  -> l = 1   | mi=4 l=1
i=0 nums[i]=  2 | nums[i] <= mi -> mi = 2  | mi=2 l=1
l = 1, r = 5 -> độ dài 5


Trace tiếp một ví dụ **có số trùng**: `nums = [1,3,3,2,2,5]`. Sau khi sắp xếp cả mảng ta có `[1,2,2,3,3,5]`, khác `nums` ở các vị trí `1,2,3,4` nên kỳ vọng độ dài **4** (đoạn `[3,3,2,2]`). Hãy xem hai số `3` và hai số `2` được xử lý ra sao: số bằng `mx` (ở `i = 2`, `3 == 3`) **không** bị coi là sai chỗ.

In [22]:
assert oracle([1, 3, 3, 2, 2, 5]) == 4
assert trace_two_pass([1, 3, 3, 2, 2, 5]) == 4

nums = [1, 3, 3, 2, 2, 5]
--- lượt 1: trái -> phải ---
i=0 nums[i]=  1 | nums[i] >= mx -> mx = 1  | mx=1 r=-1
i=1 nums[i]=  3 | nums[i] >= mx -> mx = 3  | mx=3 r=-1
i=2 nums[i]=  3 | nums[i] >= mx -> mx = 3  | mx=3 r=-1
i=3 nums[i]=  2 | nums[i] < mx  -> r = 3   | mx=3 r=3
i=4 nums[i]=  2 | nums[i] < mx  -> r = 4   | mx=3 r=4
i=5 nums[i]=  5 | nums[i] >= mx -> mx = 5  | mx=5 r=4
--- lượt 2: phải -> trái ---
i=5 nums[i]=  5 | nums[i] <= mi -> mi = 5  | mi=5 l=-1
i=4 nums[i]=  2 | nums[i] <= mi -> mi = 2  | mi=2 l=-1
i=3 nums[i]=  2 | nums[i] <= mi -> mi = 2  | mi=2 l=-1
i=2 nums[i]=  3 | nums[i] > mi  -> l = 2   | mi=2 l=2
i=1 nums[i]=  3 | nums[i] > mi  -> l = 1   | mi=2 l=1
i=0 nums[i]=  1 | nums[i] <= mi -> mi = 1  | mi=1 l=1
l = 1, r = 4 -> độ dài 4


Bây giờ trace **ngăn xếp đơn điệu** (cách 3) trên ví dụ 1, in nội dung ngăn xếp sau mỗi bước.

Lượt 1 (tìm `l`): ngăn xếp lưu **chỉ số**; pop khi giá trị ở đỉnh **lớn hơn** `nums[i]`:

| i | nums[i] | các pop | stack sau (chỉ số) | l |
|---|---|---|---|---|
| 0 | 2 | — | [0] | 7 (=n, chưa có) |
| 1 | 6 | — | [0,1] | 7 |
| 2 | 4 | pop 1 (6>4) | [0,2] | 1 |
| 3 | 8 | — | [0,2,3] | 1 |
| 4 | 10 | — | [0,2,3,4] | 1 |
| 5 | 9 | pop 4 (10>9) | [0,2,3,5] | 1 |
| 6 | 15 | — | [0,2,3,5,6] | 1 |

In [23]:
def trace_stack(nums):
    n = len(nums)
    print("nums =", nums)
    l, r = n, -1
    stk = []
    print("--- lượt 1: tìm l ---")
    for i in range(n):
        popped = []
        while stk and nums[stk[-1]] > nums[i]:
            p = stk.pop()
            popped.append(p)
            l = min(l, p)
        stk.append(i)
        print(f"i={i} nums[i]={nums[i]:>3} | pop {popped} | stack={stk} | l={l}")
    stk = []
    print("--- lượt 2: tìm r ---")
    for i in range(n - 1, -1, -1):
        popped = []
        while stk and nums[stk[-1]] < nums[i]:
            p = stk.pop()
            popped.append(p)
            r = max(r, p)
        stk.append(i)
        print(f"i={i} nums[i]={nums[i]:>3} | pop {popped} | stack={stk} | r={r}")
    ans = 0 if r == -1 else r - l + 1
    print(f"l = {l}, r = {r} -> độ dài {ans}")
    return ans

assert trace_stack([2, 6, 4, 8, 10, 9, 15]) == 5

nums = [2, 6, 4, 8, 10, 9, 15]
--- lượt 1: tìm l ---
i=0 nums[i]=  2 | pop [] | stack=[0] | l=7
i=1 nums[i]=  6 | pop [] | stack=[0, 1] | l=7
i=2 nums[i]=  4 | pop [1] | stack=[0, 2] | l=1
i=3 nums[i]=  8 | pop [] | stack=[0, 2, 3] | l=1
i=4 nums[i]= 10 | pop [] | stack=[0, 2, 3, 4] | l=1
i=5 nums[i]=  9 | pop [4] | stack=[0, 2, 3, 5] | l=1
i=6 nums[i]= 15 | pop [] | stack=[0, 2, 3, 5, 6] | l=1
--- lượt 2: tìm r ---
i=6 nums[i]= 15 | pop [] | stack=[6] | r=-1
i=5 nums[i]=  9 | pop [] | stack=[6, 5] | r=-1
i=4 nums[i]= 10 | pop [5] | stack=[6, 4] | r=5
i=3 nums[i]=  8 | pop [] | stack=[6, 4, 3] | r=5
i=2 nums[i]=  4 | pop [] | stack=[6, 4, 3, 2] | r=5
i=1 nums[i]=  6 | pop [2] | stack=[6, 4, 3, 1] | r=5
i=0 nums[i]=  2 | pop [] | stack=[6, 4, 3, 1, 0] | r=5
l = 1, r = 5 -> độ dài 5


Nhìn kết quả: ở lượt 1, `l` chốt ở `1` ngay khi `4` làm `6` (chỉ số 1) bị pop; về sau `10` (chỉ số 4) bị pop bởi `9` nhưng `min(1, 4)` vẫn là `1`. Ở lượt 2 (duyệt ngược), `10` (chỉ số 4) làm `9` (chỉ số 5) bị pop nên `r = 5`; về sau `6` làm `4` (chỉ số 2) bị pop nhưng `max(5, 2)` vẫn là `5`. Mỗi lượt chỉ cần giữ **một** phép `min` hoặc `max` trên các chỉ số bị pop.

## 9. Code nộp LeetCode

Ta nộp **cách 4** (hai lượt quét, O(n) thời gian, O(1) bộ nhớ). Trên LeetCode, `List` đã được import sẵn; trong notebook ta import ở đầu.

In [24]:
class Solution:
    def findUnsortedSubarray(self, nums: List[int]) -> int:
        n = len(nums)
        inf = float('inf')
        mx, r = -inf, -1
        for i in range(n):
            if nums[i] < mx:
                r = i
            else:
                mx = nums[i]
        mi, l = inf, -1
        for i in range(n - 1, -1, -1):
            if nums[i] > mi:
                l = i
            else:
                mi = nums[i]
        return 0 if r == -1 else r - l + 1


sol = Solution()
for nums, expected in examples:
    got = sol.findUnsortedSubarray(nums)
    print(nums, "->", got, "(kỳ vọng", expected, ")")
    assert got == expected

[2, 6, 4, 8, 10, 9, 15] -> 5 (kỳ vọng 5 )
[1, 2, 3, 4] -> 0 (kỳ vọng 0 )
[1] -> 0 (kỳ vọng 0 )


## 10. Vì sao đúng

Gọi `S` là đoạn `[l..r]` ngắn nhất cần tìm.

**Bước 1 — Nếu `i` là đầu của một cặp nghịch thế thì `i` buộc nằm trong `S`, và đầu kia cũng thế.** Giả sử `i < j`, `nums[i] > nums[j]`.

- Nếu cả `i` và `j` nằm ngoài `S`: sắp xếp `S` không đổi chỗ của chúng, thứ tự `nums[i]` đứng trước `nums[j]` với `nums[i] > nums[j]` vẫn còn → mảng chưa tăng dần. Mâu thuẫn. Vậy ít nhất một trong hai nằm trong `S`.
- Nếu `i` nằm trong `S` nhưng `j > r` nằm ngoài: sau khi sắp xếp, đoạn `S` có max `≥ nums[i] > nums[j]`; phần tử max đứng trong `S` (trước `j`) lớn hơn `nums[j]` → vẫn nghịch thế. Mâu thuẫn. Vậy `j` cũng thuộc `S`.
- Đối xứng nếu `j` nằm trong `S`, `i < l` ngoài: min của `S` `≤ nums[j] < nums[i]`, `nums[i]` đứng trước một số nhỏ hơn. Mâu thuẫn.

Kết luận: mọi cặp nghịch thế có **cả hai đầu** thuộc `S`. Do đó `S ⊇ [min i .. max j]`.

**Bước 2 — `[min i .. max j]` đủ dùng.** Gọi `L = min i`, `R = max j` trên mọi cặp nghịch thế. Phần tử ngoài `[L..R]` không tham gia cặp nghịch thế nào: bên trái `L` mọi phần tử `≤` mọi phần tử đứng sau nó, bên phải `R` tương tự. Sau khi sắp xếp `[L..R]`, mọi cặp nghịch thế cũ được giải quyết; cặp mới (một đầu trong, một đầu ngoài) không có vì các phần tử ngoài đã "không nghịch" với mọi phần tử đứng sau/trước. Vậy mảng tăng dần.

**Bước 3 — Liên hệ với `mx`, `mi`.**

- `nums[j] < max(nums[0..j-1])` đúng khi và chỉ khi `j` là đầu **phải** của một cặp nghịch thế. Lượt 1 đặt `r = j` cho `j` cuối cùng như vậy → `r = max j`.
- `nums[i] > min(nums[i+1..n-1])` đúng khi và chỉ khi `i` là đầu **trái** của một cặp nghịch thế. Lượt 2 cho `l = min i`.
- Nếu không có cặp nghịch thế nào thì `r = -1`, mảng đã không giảm → `0`.

**Về số trùng:** với `nums[j] == mx` thì `j` **không** là đầu phải (cặp `(mx, nums[j])` không phải nghịch thế vì cho phép bằng nhau), nên điều kiện phải là `<` nghiêm ngặt.

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Cặp nghịch thế | O(n²) | O(1) | vài chục triệu phép so sánh khi n = 10^4 |
| 2. Sắp xếp + so sánh | O(n log n) | O(n) | sắp xếp chi phối |
| 3. Ngăn xếp đơn điệu | O(n) | O(n) | mỗi chỉ số push/pop một lần mỗi lượt |
| 4. Max trái / Min phải | O(n) | O(1) | hai lượt quét, vài biến |
| 5. Đoạn lõm rồi nở ra | O(n) | O(n) hoặc O(1) | O(1) nếu không dùng lát cắt |

Với `n ≤ 10^4` mọi cách ngoài cách 1 đều rất nhanh. Cách 4 đạt follow up O(n) thời gian, O(1) bộ nhớ.

## 12. Bộ test

Ta tạo một danh sách ca test (ví dụ đề, biên, phản ví dụ cho các suy nghĩ sai), rồi so **cả 5 cách** với `oracle` bằng `assert`.

In [25]:
all_ways = {
    "brute_force": brute_force,
    "sort_compare": sort_compare,
    "stack_way": stack_way,
    "two_pass": two_pass,
    "dip_expand": dip_expand,
}

tests = [
    # ví dụ đề
    ([2, 6, 4, 8, 10, 9, 15], 5),
    ([1, 2, 3, 4], 0),
    ([1], 0),
    # biên
    ([2, 1], 2),                       # hai phần tử đảo
    ([1, 2], 0),
    ([1, 1, 1, 1], 0),                 # toàn số bằng nhau
    ([5, 4, 3, 2, 1], 5),              # giảm hẳn: cả mảng
    ([1, 3, 2, 2, 2], 4),
    ([1, 2, 3, 3, 3], 0),              # trùng ở cuối nhưng vẫn không giảm
    ([-1, -2, -3], 3),                 # số âm
    ([1, 2, 5, 3, 4], 3),              # phản ví dụ cho "chỉ nhìn cặp kề"
    ([1, 3, 5, 2, 6, 4, 7], 5),        # cần "nở" cả hai phía
    ([1, 3, 3, 2, 2, 5], 4),           # số trùng ở giữa
    ([2, 3, 3, 2, 4], 3),
    ([1, 2, 4, 5, 3], 3),
    ([3, 2, 1, 4, 5], 3),
    ([1, 2, 3, 5, 4], 2),
]
for nums, expected in tests:
    assert oracle(nums) == expected, (nums, oracle(nums), expected)
    for name, fn in all_ways.items():
        assert fn(list(nums)) == expected, (name, nums)
print("OK:", len(tests), "ca cố định, 5 cách khớp oracle")

OK: 17 ca cố định, 5 cách khớp oracle


### Test ngẫu nhiên nhỏ (so với oracle vét cạn)

Dùng seed cố định. Giá trị nhỏ (`-4..4`) để **nhiều số trùng**, độ dài `1..10` (đúng ràng buộc `1 <= n`).

In [26]:
random.seed(581)
cnt = 0
for _ in range(3000):
    n = random.randint(1, 10)
    nums = [random.randint(-4, 4) for _ in range(n)]
    want = oracle(nums)
    for name, fn in all_ways.items():
        got = fn(list(nums))
        assert got == want, (name, nums, got, want)
    cnt += 1
print("OK:", cnt, "mảng ngẫu nhiên nhỏ")

OK: 3000 mảng ngẫu nhiên nhỏ


### Test ngẫu nhiên lớn hơn (so các cách với nhau)

Mảng gần đã sắp xếp (rồi xáo trộn vài chỗ) để đoạn đáp án đa dạng; giá trị trong khoảng `[-10^5, 10^5]`. `oracle` quá chậm cho độ dài này nên ta đối chiếu cách 1 (cặp nghịch thế) và cách 2 (sắp xếp) — hai cách có ý tưởng độc lập — với ba cách còn lại.

In [27]:
random.seed(5810)

def near_sorted(n):
    a = sorted(random.randint(-10**5, 10**5) for _ in range(n))
    for _ in range(random.randint(0, 4)):
        i, j = random.randrange(n), random.randrange(n)
        a[i], a[j] = a[j], a[i]
    return a

cnt = 0
for _ in range(300):
    n = random.randint(1, 200)
    nums = near_sorted(n)
    ref = sort_compare(list(nums))
    assert brute_force(list(nums)) == ref
    for name in ("stack_way", "two_pass", "dip_expand"):
        assert all_ways[name](list(nums)) == ref, (name, nums)
    cnt += 1
print("OK:", cnt, "mảng gần-đã-sắp-xếp dài tới 200")

OK: 300 mảng gần-đã-sắp-xếp dài tới 200


### Test tối đa `n = 10^4`

Chỉ chạy các cách O(n) / O(n log n) (cách 1 là O(n²) có thể mất vài giây nên bỏ qua). Ba ca: đã sắp xếp, giảm hẳn, ngẫu nhiên hoàn toàn.

In [28]:
random.seed(7)
n = 10**4
big_sorted = list(range(n))
big_desc = list(range(n, 0, -1))
big_rand = [random.randint(-10**5, 10**5) for _ in range(n)]
for nums, expected in [(big_sorted, 0), (big_desc, n)]:
    for name in ("sort_compare", "stack_way", "two_pass", "dip_expand"):
        assert all_ways[name](nums) == expected, name
ref = sort_compare(big_rand)
for name in ("stack_way", "two_pass", "dip_expand"):
    assert all_ways[name](big_rand) == ref
print("OK: n = 10^4, đáp án ngẫu nhiên lớn =", ref)

t0 = time.perf_counter()
Solution().findUnsortedSubarray(big_rand)
print("hai lượt quét cho n = 10^4 chạy dưới 1 giây:", time.perf_counter() - t0 < 1.0)

OK: n = 10^4, đáp án ngẫu nhiên lớn = 10000
hai lượt quét cho n = 10^4 chạy dưới 1 giây: True


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ nhìn các cặp **kề nhau**

Nhiều bạn nghĩ: "chỗ nào `nums[i] > nums[i+1]` thì đó là đoạn lỗi", rồi lấy từ chỗ giảm đầu tiên đến chỗ giảm cuối cùng. Với `[1,2,5,3,4]` chỗ giảm duy nhất là `5 > 3` nên ra đoạn `[5,3]` (độ dài 2) — nhưng sắp xếp `[5,3]` cho `[1,2,3,5,4]`, **chưa** tăng dần. Đúng là `[5,3,4]` (độ dài 3).

In [29]:
def wrong_adjacent(nums):
    n = len(nums)
    l, r = -1, -1
    for i in range(n - 1):
        if nums[i] > nums[i + 1]:
            if l == -1:
                l = i
            r = i + 1
    return 0 if l == -1 else r - l + 1

nums = [1, 2, 5, 3, 4]
print("sai    :", wrong_adjacent(nums), "| đúng:", oracle(nums))
assert wrong_adjacent(nums) != oracle(nums)

sai    : 2 | đúng: 3


**Vì sao sai?** Phần tử `4` đứng **sau** `3` nhưng vẫn nhỏ hơn `5` đứng xa hơn — quan hệ nghịch thế không chỉ xảy ra giữa hai vị trí kề nhau. Cần so với `max` bên trái / `min` bên phải. Code đúng: `two_pass` ở mục 7.3.

In [30]:
assert two_pass([1, 2, 5, 3, 4]) == 3
print("đúng:", two_pass([1, 2, 5, 3, 4]))

đúng: 3


### Lỗi 2 — Khởi tạo `l = r = 0` rồi trả `r - l + 1`

Nếu mảng đã sắp xếp, không có lần cập nhật nào, `l = r = 0` → trả `0 - 0 + 1 = 1`, nhưng đáp án là `0`. Phải dùng giá trị canh `-1` (hoặc `n` / `-1`) và xử lý riêng "không cập nhật".

In [31]:
def wrong_init(nums):
    n = len(nums)
    inf = float('inf')
    mx, r = -inf, 0                    # SAI: r = 0
    for i in range(n):
        if nums[i] < mx:
            r = i
        else:
            mx = nums[i]
    mi, l = inf, 0                     # SAI: l = 0
    for i in range(n - 1, -1, -1):
        if nums[i] > mi:
            l = i
        else:
            mi = nums[i]
    return r - l + 1

print("sai    :", wrong_init([1, 2, 3, 4]), "| đúng:", oracle([1, 2, 3, 4]))
assert wrong_init([1, 2, 3, 4]) == 1
assert two_pass([1, 2, 3, 4]) == 0

sai    : 1 | đúng: 0


**Sửa:** `r = -1`, `l = -1`, và cuối hàm `return 0 if r == -1 else r - l + 1`.

### Lỗi 3 — Dùng `<=` / `>=` sai khi có số trùng

Nếu ở lượt 1 viết `if nums[i] <= mx: r = i`, thì với `[1,2,2,3]` phần tử `2` thứ hai bằng `mx = 2` bị coi là sai chỗ → trả đoạn dài hơn 0 trong khi mảng đã không giảm.

In [32]:
def wrong_equal(nums):
    n = len(nums)
    inf = float('inf')
    mx, r = -inf, -1
    for i in range(n):
        if nums[i] <= mx:              # SAI: phải là <
            r = i
        else:
            mx = nums[i]
    mi, l = inf, -1
    for i in range(n - 1, -1, -1):
        if nums[i] >= mi:              # SAI: phải là >
            l = i
        else:
            mi = nums[i]
    return 0 if r == -1 else r - l + 1

print("sai    :", wrong_equal([1, 2, 2, 3]), "| đúng:", oracle([1, 2, 2, 3]))
assert wrong_equal([1, 2, 2, 3]) != 0
assert two_pass([1, 2, 2, 3]) == 0

sai    : 2 | đúng: 0


### Lỗi 4 — Sắp xếp tại chỗ rồi so sánh với chính nó

`nums.sort()` làm đổi `nums` và trả `None`. Nếu viết `arr = nums.sort()` thì `arr` là `None`; nếu viết `nums.sort()` rồi so `nums[l] == nums[l]` thì luôn bằng nhau → trả `0` hoặc lỗi. Phải sao chép trước: `arr = sorted(nums)`.

In [33]:
def wrong_inplace(nums):
    n = len(nums)
    arr = nums
    arr.sort()                         # SAI: sắp xếp luôn cả nums
    l, r = 0, n - 1
    while l <= r and nums[l] == arr[l]:
        l += 1
    while l <= r and nums[r] == arr[r]:
        r -= 1
    return r - l + 1

print("sai    :", wrong_inplace([2, 6, 4, 8, 10, 9, 15]), "| đúng: 5")
assert wrong_inplace([2, 6, 4, 8, 10, 9, 15]) == 0
assert sort_compare([2, 6, 4, 8, 10, 9, 15]) == 5

sai    : 0 | đúng: 5


### Lỗi 5 — Cách 5 mà quên bước "nở ra"

Chỉ lấy đoạn lõm `[i..j]` giữa chỗ giảm đầu tiên và cuối cùng rồi dừng. Với `[1,3,5,2,6,4,7]`: chỗ giảm đầu tiên là `5 > 2` (i = 2), cuối cùng `6 > 4` (j = 5) → đoạn dài 4; nhưng `3 > 2` (min của đoạn là `2`) nên `3` cũng phải nằm trong đoạn → đúng là 5.

In [34]:
def wrong_no_expand(nums):
    n = len(nums)
    i = 0
    while i + 1 < n and nums[i] <= nums[i + 1]:
        i += 1
    if i == n - 1:
        return 0
    j = n - 1
    while j > 0 and nums[j - 1] <= nums[j]:
        j -= 1
    return j - i + 1                   # SAI: thiếu bước nở ra

nums = [1, 3, 5, 2, 6, 4, 7]
print("sai    :", wrong_no_expand(nums), "| đúng:", oracle(nums))
assert wrong_no_expand(nums) == 4 and dip_expand(nums) == 5

sai    : 4 | đúng: 5


## 14. Mẹo & nhận dạng pattern

- **"Đoạn ngắn nhất để sắp xếp cho cả mảng đúng"** → nghĩ ngay tới `max` chạy từ trái và `min` chạy từ phải, hoặc "so với bản đã sắp xếp".
- **Phần tử đúng chỗ ⇔ `max(trái) <= x <= min(phải)`**: mẹo kiểm tra tại chỗ bằng hai biến, không cần mảng phụ.
- **Hai biên thường tìm ở hai lượt quét ngược chiều nhau** (một lượt giữ max, lượt kia giữ min) — mẫu chung với bài "Trapping Rain Water" (max trái / max phải), "Product of Array Except Self" (tích tiền tố / hậu tố).
- **Ngăn xếp đơn điệu** thường xuất hiện khi bài hỏi "phần tử lớn/nhỏ hơn gần nhất" — ở đây pop xảy ra đúng khi phát hiện cặp nghịch thế.
- **Số trùng**: luôn tự hỏi "bằng nhau có tính là sai thứ tự không?" rồi chọn `<` hay `<=` cho khớp.
- **Kiểm chứng**: viết `oracle` vét cạn (thử mọi đoạn) và so với lời giải nhanh trên mảng nhỏ ngẫu nhiên có nhiều số trùng — bắt lỗi dấu rất hiệu quả.
- Với `n ≤ 10^4` cách "sắp xếp rồi so sánh" ngắn nhất, dễ nhớ nhất; nếu phỏng vấn hỏi O(n) thì chuyển sang hai lượt quét.

## 15. Biến thể

### 15.1. Trả về cả hai biên `[l, r]` (hoặc `[-1, -1]` nếu đã sắp xếp)

Chỉ cần đổi dòng `return`.

In [35]:
def find_bounds(nums):
    n = len(nums)
    inf = float('inf')
    mx, r = -inf, -1
    for i in range(n):
        if nums[i] < mx:
            r = i
        else:
            mx = nums[i]
    mi, l = inf, -1
    for i in range(n - 1, -1, -1):
        if nums[i] > mi:
            l = i
        else:
            mi = nums[i]
    return [-1, -1] if r == -1 else [l, r]

print(find_bounds([2, 6, 4, 8, 10, 9, 15]))
print(find_bounds([1, 2, 3, 4]))
assert find_bounds([2, 6, 4, 8, 10, 9, 15]) == [1, 5]
assert find_bounds([1, 2, 3, 4]) == [-1, -1]

[1, 5]
[-1, -1]


### 15.2. Sắp xếp thành thứ tự **giảm dần** (không tăng)

Đổi vai trò dấu: một phần tử đúng chỗ khi `min(trái) >= x >= max(phải)`. Cách đơn giản nhất là **đảo dấu mọi số** (`-x`) rồi gọi lời giải gốc: giảm dần với `x` chính là tăng dần với `-x`.

In [36]:
def unsorted_desc(nums):
    return two_pass([-x for x in nums])

print(unsorted_desc([9, 5, 7, 6, 2, 1]))   # đoạn [5,7,6] -> 3
assert unsorted_desc([9, 5, 7, 6, 2, 1]) == 3
assert unsorted_desc([5, 4, 3]) == 0

3


### 15.3. Phiên bản O(1) bộ nhớ của cách 5

Thay `min(seg)`/`max(seg)` bằng một vòng `for` chạy từ `i` đến `j` để tránh tạo lát cắt. Đây là bài tập tự luyện nhỏ ở mục 17.

### 15.4. Bài liên quan

- Bài "đoạn tối thiểu cần sắp xếp" (*Sub Sort*) trong sách *Cracking the Coding Interview* dùng cùng ý tưởng min/max này.
- **Monotonic stack**: LeetCode 496 *Next Greater Element I*, 739 *Daily Temperatures* — dùng chung kỹ thuật pop khi gặp phần tử vi phạm thứ tự.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Cặp nghịch thế | `l = min i`, `r = max j` trên mọi cặp `nums[i] > nums[j]` | O(n²) | O(1) | để hiểu bản chất; kiểm chứng; n nhỏ |
| 2. Sắp xếp + so sánh | đoạn = vị trí khác đầu tiên và cuối cùng so với bản đã sắp xếp | O(n log n) | O(n) | cách dễ nhất, đủ cho n ≤ 10^4 |
| 3. Ngăn xếp đơn điệu | pop khi gặp nghịch thế, lấy min/max chỉ số bị pop | O(n) | O(n) | khi cần luyện kỹ thuật monotonic stack |
| 4. Max trái / Min phải | `nums[i] < max trái` ⇒ `r = i`; `nums[i] > min phải` ⇒ `l = i` | O(n) | O(1) | **nên dùng**: nhanh, gọn, đạt follow up |
| 5. Đoạn lõm rồi nở | tìm chỗ giảm đầu/cuối, lấy min/max rồi mở rộng | O(n) | O(1)–O(n) | thích hợp giải thích bằng lời "như người" |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell **TODO** chỉ gán `None` hoặc định nghĩa hàm rỗng — **không được gọi** trước khi bạn viết xong. Đáp án ở mục 19.

### Bài 1 (đoán output)

Không chạy code, hãy đoán `result` của đoạn dưới rồi điền vào `my_guess_1`.

In [37]:
nums = [1, 5, 3, 4, 2, 6]
srt = sorted(nums)
diff = [i for i in range(len(nums)) if nums[i] != srt[i]]
result = diff[-1] - diff[0] + 1
# print(result)   # <- bỏ dấu # sau khi đã đoán xong để kiểm tra

my_guess_1 = None    # TODO: điền số bạn đoán

### Bài 2 (tính tay)

Với `nums = [1, 3, 2, 2, 2]`, hãy tính **bằng tay** hai giá trị `l` và `r` theo cách "max trái / min phải" (nhớ rằng số trùng nên dùng `<` và `>` nghiêm ngặt), rồi suy ra độ dài. Điền vào các biến.

In [38]:
my_l = None       # TODO: biên trái
my_r = None       # TODO: biên phải
my_len = None     # TODO: độ dài

### Bài 3 (điền TODO)

Hoàn thành hàm hai lượt quét dưới đây: thay `...` bằng điều kiện/phép gán đúng.

In [39]:
def exercise3(nums):
    n = len(nums)
    inf = float('inf')
    mx, r = -inf, -1
    for i in range(n):
        if ...:                      # TODO: điều kiện "có số lớn hơn nằm bên trái"
            ...                      # TODO: cập nhật r
        else:
            mx = nums[i]
    mi, l = inf, -1
    for i in range(n - 1, -1, -1):
        if ...:                      # TODO: điều kiện "có số nhỏ hơn nằm bên phải"
            ...                      # TODO: cập nhật l
        else:
            mi = nums[i]
    return 0 if r == -1 else r - l + 1

### Bài 4 (tự viết lại)

Tự viết hàm `exercise4(nums)` giải bài bằng **sắp xếp + so sánh** (không nhìn lại cách 2), trả về độ dài đoạn ngắn nhất.

In [40]:
def exercise4(nums):
    # TODO: viết lời giải
    pass

### Bài 5 (biến thể / mini challenge)

Viết hàm `exercise5(nums)` trả về **số phần tử đã đúng chỗ nhưng vẫn phải nằm trong đoạn cần sắp xếp** (tức là độ dài đoạn trừ đi số vị trí mà `nums[i] != sorted(nums)[i]`). Ví dụ `[2,6,4,8,10,9,15]` → đoạn dài 5, số vị trí khác `sorted` là 4 (`6,4,10,9`) → trả `1` (phần tử `8`).

In [41]:
def exercise5(nums):
    # TODO: viết lời giải
    pass

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: viết `sorted(nums)` ra giấy và so từng vị trí.
- Mức 2: những vị trí khác nhau tạo list `diff`; chỉ cần phần tử đầu và cuối.
- Mức 3: `nums = [1,5,3,4,2,6]` có `sorted = [1,2,3,4,5,6]`; so từng chỉ số: chỉ số `2` (3 vs 3) và `3` (4 vs 4) giống nhau, vậy hai chỉ số khác nhau là những chỉ số nào? Đáp án là hiệu của chúng cộng 1.

**Bài 2**
- Mức 1: duyệt trái → phải ghi `mx` sau mỗi bước; đánh dấu bước `nums[i] < mx`.
- Mức 2: `r` là vị trí **cuối cùng** bị đánh dấu; `l` là vị trí **đầu tiên** (khi duyệt phải → trái, là lần cập nhật **cuối**) có `nums[i] > mi`.
- Mức 3: với `[1,3,2,2,2]`, các số `2` đều nhỏ hơn `3` (đứng trước chúng) nên `r = 4`; `3` lớn hơn min bên phải `2` nên `l = 1`.

**Bài 3**
- Mức 1: nhìn lại bảng trace mục 8.
- Mức 2: điều kiện lượt 1 so `nums[i]` với `mx`; lượt 2 so với `mi`.
- Mức 3: `nums[i] < mx` → `r = i`; `nums[i] > mi` → `l = i`.

**Bài 4**
- Mức 1: tạo bản đã sắp xếp bằng `sorted`.
- Mức 2: hai con trỏ `l`, `r` đi từ hai đầu vào trong khi hai mảng bằng nhau.
- Mức 3: nhớ điều kiện `l <= r` để mảng đã sắp xếp trả `0`.

**Bài 5**
- Mức 1: độ dài đoạn tính bằng một trong các cách đã học.
- Mức 2: đếm `sum(1 for a, b in zip(nums, sorted(nums)) if a != b)`.
- Mức 3: kết quả = độ dài − số vị trí khác; nếu mảng đã sắp xếp, cả hai đều `0`.

## 19. Đáp án tham khảo

Chạy các cell sau để kiểm tra bằng `assert`. Bài 2 có đáp án tự điền ở trên nên so với đáp án mẫu ở đây (không đụng tới `my_*`).

In [42]:
# Bài 1
nums = [1, 5, 3, 4, 2, 6]
srt = sorted(nums)
diff = [i for i in range(len(nums)) if nums[i] != srt[i]]
answer_1 = diff[-1] - diff[0] + 1
print("Bài 1:", answer_1, "| các vị trí khác:", diff)
assert answer_1 == 4

# Bài 2: nums = [1,3,2,2,2]
nums2 = [1, 3, 2, 2, 2]
inf = float('inf')
mx, r2 = -inf, -1
for i, x in enumerate(nums2):
    if x < mx:
        r2 = i
    else:
        mx = x
mi, l2 = inf, -1
for i in range(len(nums2) - 1, -1, -1):
    if nums2[i] > mi:
        l2 = i
    else:
        mi = nums2[i]
print("Bài 2: l =", l2, ", r =", r2, ", độ dài =", r2 - l2 + 1)
assert (l2, r2, r2 - l2 + 1) == (1, 4, 4)
assert oracle(nums2) == 4

Bài 1: 4 | các vị trí khác: [1, 4]
Bài 2: l = 1 , r = 4 , độ dài = 4


In [43]:
# Bài 3
def exercise3_solution(nums):
    n = len(nums)
    inf = float('inf')
    mx, r = -inf, -1
    for i in range(n):
        if nums[i] < mx:
            r = i
        else:
            mx = nums[i]
    mi, l = inf, -1
    for i in range(n - 1, -1, -1):
        if nums[i] > mi:
            l = i
        else:
            mi = nums[i]
    return 0 if r == -1 else r - l + 1

# Bài 4
def exercise4_solution(nums):
    arr = sorted(nums)
    l, r = 0, len(nums) - 1
    while l <= r and nums[l] == arr[l]:
        l += 1
    while l <= r and nums[r] == arr[r]:
        r -= 1
    return r - l + 1

# Bài 5
def exercise5_solution(nums):
    length = exercise4_solution(nums)
    wrong = sum(1 for a, b in zip(nums, sorted(nums)) if a != b)
    return length - wrong

for nums in ([2, 6, 4, 8, 10, 9, 15], [1, 2, 3, 4], [1], [5, 4, 3, 2, 1], [1, 3, 3, 2, 2, 5]):
    assert exercise3_solution(nums) == oracle(nums)
    assert exercise4_solution(nums) == oracle(nums)
assert exercise5_solution([2, 6, 4, 8, 10, 9, 15]) == 1
assert exercise5_solution([1, 2, 3, 4]) == 0
assert exercise5_solution([2, 1]) == 0
print("Đáp án bài 3, 4, 5 đúng")

random.seed(19)
for _ in range(500):
    nums = [random.randint(-3, 3) for _ in range(random.randint(1, 9))]
    assert exercise3_solution(nums) == exercise4_solution(nums) == oracle(nums)
print("Đối chiếu ngẫu nhiên OK")

Đáp án bài 3, 4, 5 đúng
Đối chiếu ngẫu nhiên OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (hai lượt quét, O(n) thời gian, O(1) bộ nhớ):**

```python
class Solution:
    def findUnsortedSubarray(self, nums: List[int]) -> int:
        n = len(nums)
        inf = float('inf')
        mx, r = -inf, -1
        for i in range(n):              # trái -> phải
            if nums[i] < mx:
                r = i                   # có số lớn hơn nằm bên trái
            else:
                mx = nums[i]
        mi, l = inf, -1
        for i in range(n - 1, -1, -1):  # phải -> trái
            if nums[i] > mi:
                l = i                   # có số nhỏ hơn nằm bên phải
            else:
                mi = nums[i]
        return 0 if r == -1 else r - l + 1
```

**Checklist trước khi submit:**

- [ ] `l`, `r` khởi tạo `-1`, và trả `0` khi `r == -1` (mảng đã sắp xếp).
- [ ] So sánh **nghiêm ngặt** (`<`, `>`), vì số trùng không phải nghịch thế.
- [ ] Lượt trái → phải giữ **max** và cập nhật `r`; lượt phải → trái giữ **min** và cập nhật `l`.
- [ ] Không sắp xếp tại chỗ làm đổi `nums` (nếu dùng cách sắp xếp, dùng `sorted`).
- [ ] Đã thử: `[1]`, `[1,1,1]`, `[2,1]`, `[5,4,3,2,1]`, `[1,2,5,3,4]`, `[1,3,2,2,2]`.
- [ ] Đã so với `oracle` trên mảng nhỏ ngẫu nhiên có nhiều số trùng.
- [ ] Độ phức tạp: Time O(n), Space O(1) (đạt follow up).